# temperature와 샘플링

같은 프롬프트에서 temperature를 바꾸면 반복 출력의 다양성이 어떻게 달라질까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 로컬 생성 모델 준비

고정 revision의 사전학습 모델을 불러옵니다. `generate()`는 프롬프트와 temperature, 난수 시드, 최대 생성 토큰 수를 입력받습니다. tokenizer가 문자열을 ID로 바꾸고 모델이 새 ID를 생성하며, decode가 새 부분을 문자열로 바꾸어 반환합니다.

In [1]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, logging, set_seed

logging.set_verbosity_error()

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION)
model.eval()
print("model:", MODEL_ID)
print("parameters:", f"{model.num_parameters():,}")

model: Qwen/Qwen2.5-0.5B-Instruct
parameters: 494,032,768


In [2]:
def generate(prompt: str, *, temperature: float = 0.7, seed: int = 0, max_new_tokens: int = 48) -> str:
    # 채팅 템플릿을 적용한 뒤 새로 생성된 토큰만 복원합니다.
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    set_seed(seed)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9 if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 로짓을 확률로 바꾸기

로짓은 모델이 다음 토큰 후보마다 계산한 변환 전 점수입니다. 아래의 세 값은 설명을 위해 정한 예이며 실제 Qwen에서 추출한 값이 아닙니다. 로짓을 temperature로 나눈 뒤 softmax를 적용하면 합이 1인 확률이 반환됩니다. 두 출력에서 가장 큰 후보는 유지되는지, 후보 사이의 확률 차이는 어떻게 달라지는지 읽습니다.

In [3]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
toy_logits = torch.tensor([2.0, 1.0, 0.0])
for temperature in [0.5, 1.0]:
    probabilities = torch.softmax(toy_logits / temperature, dim=0)
    print(f"T={temperature:.1f} probabilities:", [round(x, 4) for x in probabilities.tolist()])
    print("합계:", round(float(probabilities.sum()), 4))

T=0.5 probabilities: [0.8668, 0.1173, 0.0159]
합계: 1.0
T=1.0 probabilities: [0.6652, 0.2447, 0.09]
합계: 1.0


## temperature별 반복 생성

프롬프트, top-p, 최대 생성 길이와 시드 세 개는 고정하고 temperature만 0.3과 1.0으로 바꿉니다. 각 줄은 `temperature → seed → 반환 문자열` 순서입니다. 같은 시드끼리 두 temperature 결과를 대응해 읽은 뒤, 세 번 전체의 문자열 차이를 확인합니다.

In [4]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
prompt = "오프라인 서점의 가을 행사 이름을 한국어 5어절 이내로 하나 제안하세요."
records = []
for temperature in [0.3, 1.0]:
    for seed in [11, 22, 33]:
        answer = generate(prompt, temperature=temperature, seed=seed, max_new_tokens=24)
        records.append((temperature, seed, answer))
        print(f"T={temperature:.1f} seed={seed}: {answer}")

T=0.3 seed=11: "열정과 풍부한 책: 오프라인 서점의 가을 행사를 맞이


T=0.3 seed=22: 1. "열매와 함께: 가을의 풍경"
2. "오늘의 날


T=0.3 seed=33: "가을의 푸른 시선: 오프라인 서점의 봄행사"


T=1.0 seed=11: 죄송합니다, 저는 실시간 정보나 실제 위치에 맞는 인적 자원을 가지고 있지 않습니다. 하지만


T=1.0 seed=22: 1. "열린 시선" (Sunlit Eye)
2. "월드를 뚫고"


T=1.0 seed=33: "온라인 어워서행 특별 판매"


## 서로 다른 출력 수

문자열이 정확히 같은지만 세어, 표현의 다양성을 제한적으로 관찰합니다.

In [5]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
for temperature in [0.3, 1.0]:
    answers = [answer for t, _, answer in records if t == temperature]
    print(f"T={temperature:.1f} unique={len(set(answers))}/3")
print("문자열 종류 수는 제안의 품질이나 창의성을 직접 측정하지 않습니다.")

T=0.3 unique=3/3
T=1.0 unique=3/3
문자열 종류 수는 제안의 품질이나 창의성을 직접 측정하지 않습니다.


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [6]:
assert torch.isclose(torch.softmax(toy_logits, dim=0).sum(), torch.tensor(1.0))
assert len(records) == 6
assert {t for t, _, _ in records} == {0.3, 1.0}
print("검증: 로짓의 확률 변환과 2개 temperature × 동일한 3개 시드의 출력을 확인했습니다.")

검증: 로짓의 확률 변환과 2개 temperature × 동일한 3개 시드의 출력을 확인했습니다.
